# RNN用系列を非線形ARへ変換

`ISLP23-C10-Q11` · ISLP 2023 · Chapter 10, Exercise 11

## この問題で行うこと

前問Q10と同じNYSEの短い時系列窓を平坦化し、今度は非線形モデルで自己回帰する。Q10の線形ARと同じ分割でテストR²を比較する。

[原典の設問・正確な条件を確認する](https://drive.google.com/file/d/1ajFkHO6zjrdGNqhqW1jKBZdiNGh_8YQ1/view)

この説明は学習用の要約です。数値条件・指定手法・各小問は原典に従ってください。

## 0. NYSEデータを取得する

[ISLP公式のNYSEデータ](https://intro-stat-learning.github.io/ISLP/datasets/NYSE.html)を、このセルで直接読み込みます。Colabへの手動アップロードやISLPのインストールは不要です。`train` は教材が指定する訓練・テスト区分です。

In [ ]:
import numpy as np
import pandas as pd

NYSE_CSV = 'https://raw.githubusercontent.com/intro-stat-learning/ISLP/a1f4e43ca88d4c9a6f186930181d628b9270015d/ISLP/data/NYSE.csv'
nyse = pd.read_csv(NYSE_CSV, parse_dates=['date'])
required = {'date', 'DJ_return', 'log_volume', 'log_volatility', 'train'}
assert required.issubset(nyse.columns), nyse.columns.tolist()
assert nyse.shape == (6051, 6), nyse.shape
assert not nyse[list(required)].isna().any().any()
assert nyse['train'].dtype == bool
print('NYSE:', nyse.shape, 'train:', int(nyse['train'].sum()), 'test:', int((~nyse['train']).sum()))
print(nyse.head())


## 前問Q10とのつながり

Q10で作ったNYSEの短い時系列窓・目的変数・訓練/テスト分割をそのまま使います。窓を2次元の特徴行列へ平坦化した後、線形回帰の代わりに非線形回帰器を学習します。**同じテスト集合のR²**でQ10の線形ARと比較してください。時系列の未来情報が訓練側へ混ざらないよう、分割順序も維持します。

In [ ]:
# 当日より前の5営業日を特徴量にする。未来の値は入れない。
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

series_cols = ['DJ_return', 'log_volume', 'log_volatility']
windows = np.stack(
    [nyse[series_cols].shift(lag).to_numpy() for lag in range(5, 0, -1)],
    axis=1,
)
valid = np.isfinite(windows).all(axis=(1, 2))
X_seq = windows[valid]
y = nyse.loc[valid, 'log_volume'].to_numpy()
is_train = nyse.loc[valid, 'train'].to_numpy()
X_flat = X_seq.reshape(len(X_seq), -1)
scaler = StandardScaler().fit(X_flat[is_train])
X_train = scaler.transform(X_flat[is_train])
X_test = scaler.transform(X_flat[~is_train])
y_train, y_test = y[is_train], y[~is_train]
print('train/test:', X_train.shape, X_test.shape)
assert X_train.shape == (4276, 15), X_train.shape
assert X_test.shape == (1770, 15), X_test.shape


## 2. 比較の基準となる線形AR

Q10と同じ平坦化された入力で線形回帰を学習します。この値を固定してから、次の非線形モデルを評価します。

In [ ]:
from sklearn.linear_model import LinearRegression

linear_ar = LinearRegression().fit(X_train, y_train)
linear_r2 = r2_score(y_test, linear_ar.predict(X_test))
print(f'線形AR test R²: {linear_r2:.4f}')


## 3. 非線形ARを試す

下は動作する出発点です。隠れ層の幅や正則化を変更し、**訓練期間内の検証データ**だけで選んでください。テスト期間のR²はモデル決定後に確認します。

In [ ]:
from sklearn.neural_network import MLPRegressor

# TODO: 訓練期間内で構造・正則化を選び、選定理由を記録する。
nonlinear_ar = MLPRegressor(
    hidden_layer_sizes=(32,), alpha=0.1,
    max_iter=600, random_state=42,
).fit(X_train, y_train)
nonlinear_r2 = r2_score(y_test, nonlinear_ar.predict(X_test))
print(f'非線形AR test R²: {nonlinear_r2:.4f}')
print(f'差分: {nonlinear_r2 - linear_r2:+.4f}')


## 4. 比較と考察

同じ期間・特徴量で比べ、なぜ差が出たかを記録してください。訓練期間内の検証方法、ハイパーパラメータ、テスト期間の誤差も説明してみましょう。

In [ ]:
results = pd.DataFrame({
    'model': ['linear AR', 'nonlinear AR'],
    'test_R2': [linear_r2, nonlinear_r2],
})
print(results.to_string(index=False))

# TODO: 残差の時系列や、訓練期間内の検証結果を追加する。


**自分の考察：**

- 非線形化で何が改善／悪化したか：
- 訓練期間内でのモデル選択方法：
- テスト期間のR²を見てから設定を変えなかったか：
